This notebook walks through the construction of a ROC curve and verifies it against built-in `scikit-learn` functions. Then we will test our intuition for several cases.

**Overview:** In this notebook you will
- Write a function that calculates a ROC curve and verify it against built-in functions from sklearn
- Test your intuition for different cases
- (Optional) try it out on your own data!

Look for `#! YOUR CODE HERE` comments for where to add your own solution

In [ ]:
# Import libraries
import numpy as np
from numpy.random import default_rng
import sklearn.metrics as metrics

import matplotlib.pyplot as plt

NSCORES = 1000
N_classA = NSCORES//2
N_classB = NSCORES//2

In [ ]:
def process_scores(score_classA, score_classB):
  scores = np.concatenate((score_classA, score_classB))
  labels = np.concatenate((np.repeat(0, score_classA.shape[0]), np.repeat(1, score_classB.shape[0])))
  return scores, labels
  # label ==0 for class A (purple) and ==1 for class B (green)

# Get score values

In [ ]:
np.random.seed(7)
score_classA = np.random.normal(loc = 10, scale = 2, size = N_classA)
score_classB = np.random.normal(loc = 12, scale = 2, size = N_classB)

bins = np.linspace(0,20, 20)
plt.hist(score_classA, bins=bins, color='#A55194', histtype='step', linewidth=3)
plt.hist(score_classB, bins=bins, color='#8CA252', histtype='step', linewidth=3)

plt.xlabel(r"Score", fontsize=14)
plt.ylabel(r"Counts", fontsize=14)
plt.show()

In [ ]:
# Combine into a single score array and create labels
scores, labels = process_scores(score_classA, score_classB)

# Warm-up: Single threshold

Let's consider a single threshold at `T=5`. Above this threshold we will label every point as class B, however the true class B is what is shown in green.

In [ ]:
plt.scatter(scores[0:N_classA], np.ones(N_classA), color='#A55194')
plt.scatter(scores[N_classA:], np.ones(N_classB), color='#8CA252')

T = 5
plt.plot([T, T], [0.5, 1.5], color='#929292', linestyle='--')

plt.xlabel(r"Score", fontsize=14)
plt.yticks([])
plt.title("True scores")
plt.show()

The **True Positive Rate (TPR)** is the fraction of green above `T` and the **False Positive Rate** is the fraction of purple above `T`

In [ ]:
passing_mask = scores >= T                # This is what we say should be called class B

predicted_classB = scores[passing_mask]
predicted_classA = scores[~passing_mask]

In [ ]:
# Plot our classifier's prediciton based on the threshold T
plt.scatter(predicted_classA, 1.1*np.ones_like(predicted_classA), color='#A55194')
plt.scatter(predicted_classB, 1.1*np.ones_like(predicted_classB), color='#8CA252')

# Plot truth
plt.scatter(scores[0:N_classA], np.ones(N_classA), color='#A55194')
plt.scatter(scores[N_classA:], np.ones(N_classB), color='#8CA252')

# Plot the threshold
plt.plot([T, T], [0.9, 1.2], color='#929292', linestyle='--')

plt.xlabel(r"Score", fontsize=14)
plt.yticks([1, 1.1], ['Truth', 'Classifier from T'], fontsize=14)
plt.show()



Now we need to see how poorly we did. Namely, how many true purples did we call green and how many true greens did we call purple?

In [ ]:
# True Positive Rate: How many greens are above our threshold relative to the total number of green samples?
tpr = np.sum((passing_mask == True) & (labels == 1))/(N_classB)
print(tpr)

# False Positive Rate: How many purples are above our threshold relative to the total number of purple samples?
fpr = np.sum((passing_mask == True) & (labels == 0))/(N_classA)
print(fpr)

So we successfully labeled all green samples (TPR=1), but have a very high chance of mistaking a purple sample for a green one (FPR=0.992)

#### **Question:** Based on the plot above what do you think would happen if `T=5.1` instead?

#### **Answer:**

In [ ]:
T_new = 5.1
passing_mask = scores >= T_new

tpr = np.sum((passing_mask == True) & (labels == 1))/(N_classB)
fpr = np.sum((passing_mask == True) & (labels == 0))/(N_classA)
print(tpr)
print(fpr)

Nothing happened! This is because no sample points switched sides. We would have to move the threshold more to start to see a difference.

In [ ]:
T_new = 5.4
passing_mask = scores >= T_new

tpr = np.sum((passing_mask == True) & (labels == 1))/(N_classB)
fpr = np.sum((passing_mask == True) & (labels == 0))/(N_classA)
print(tpr)
print(fpr)

Therefore, it makes sense for our choices of threshold to correspond to where the unique scores fall.

In [ ]:
thresholds = np.sort(np.unique(scores))

# Build your own ROC curve!

In this section, we will put what we have learned into practice. We will first build a custom ROC curve function and then test it against `sklearn`'s built-in version

In [ ]:
def customROCCurve(labels, scores):
  """
         labels: ndarray of labels of the true type of each score value; shape=(NSAMPLES,)
         scores: ndarray of score values; shape=(NSAMPLES,)
       N_classA: Number of scores that truly belong to class A (purple); int
  """
  N_classA = int(np.sum(labels==0))
  N_classB = len(scores) - N_classA

  # Get thresholds
  #! YOUR CODE HERE
  #thresholds =

  fpr = np.ones_like(thresholds)*-999
  tpr = np.ones_like(thresholds)*-999

  for (i,t) in enumerate(thresholds):
    #! YOUR CODE HERE
    # passing_mask =
    # fpr[i] =
    # tpr[i] =

  return fpr, tpr, thresholds

We'll then call this from a wrapper function with a bit more functionality

In [ ]:
def calc_ROC_and_AUC(labels, scores, MODE='custom', INTERPOLATE=True):
  """
       labels: ndarray of labels of the true type of each score value; shape=(NSAMPLES,)
       scores: ndarray of score values; shape=(NSAMPLES,)
         MODE: Flag to decide whether to run the custom ROC curve code or sklearn's function ('custom' or 'sklearn')
  INTERPOLATE: Flag to interpolate the tpr vs fpr function. By default the values of tpr will be different
               for different samples, this interpolates to get the fpr over a standardized spacing. This is
               necessary for plotting error bands.
  """

  # Calculate FPR and TPR
  if MODE=='sklearn':
    fpr_raw, tpr_raw, _ = metrics.roc_curve(labels, scores)
  else:
    fpr_raw, tpr_raw, _ = customROCCurve(labels, scores, )

  if INTERPOLATE:
    # https://stats.stackexchange.com/questions/186337/average-roc-for-repeated-10-fold-cross-validation-with-probability-estimates
    base_tpr = np.linspace(0, 1, 101)  # 0.00, 0.01, ..., 1.0
    tpr = base_tpr
    fpr = np.interp(base_tpr, tpr_raw, fpr_raw)
  else:
    fpr = fpr_raw
    tpr = tpr_raw

  # Get other confusion matrix elements
  tnr = 1 - fpr
  fnr = 1 - tpr

  # Calculate AUC
  auc = metrics.roc_auc_score(labels, scores)

  return tpr, fpr, fnr, tnr, auc

# Let's test it out!

In [ ]:
tpr_custom, _, _, tnr_custom, _   = calc_ROC_and_AUC(labels, scores, MODE='custom', INTERPOLATE=False)
tpr_sklearn, _, _, tnr_sklearn, auc = calc_ROC_and_AUC(labels, scores, MODE='sklearn', INTERPOLATE=False)


fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(6, 6))
ax.plot(tpr_sklearn, tnr_sklearn, color='tab:blue', linewidth=3)
ax.plot(tpr_custom, tnr_custom, color='tab:orange', linewidth=3, linestyle='--')

ax.set_xlabel('True Positive Rate (TPR)', fontsize=14)
ax.set_ylabel('True Negative Rate (TNR)', fontsize=14)

plt.show()

print("AUC = ", auc)

# Testing your intuition

#### **Question:** What AUC would correspond to random guessing? What is a score distribution that would lead to this?

In [ ]:
#! YOUR CODE HERE
# score_classA =                         # shape (N_classA,)
# score_classB =                         # shape (N_classB,)

scores, labels = process_scores(score_classA, score_classB)

_, _, _, _, auc = calc_ROC_and_AUC(labels, scores, MODE='sklearn')

In [ ]:
bins = np.linspace(scores.min() -1,scores.max()+1, 40)
plt.hist(score_classA, bins=bins, color='#A55194', histtype='step', linewidth=3)
plt.hist(score_classB, bins=bins, color='#8CA252', histtype='step', linewidth=3)

plt.xlabel(r"Score", fontsize=14)
plt.ylabel(r"Counts", fontsize=14)
plt.show()

print("AUC = ", auc)

#### **Question:** What AUC would correspond to perfect classification?  What is a score distribution that would lead to this?

In [ ]:
#! YOUR CODE HERE
# score_classA =                         # shape (N_classA,)
# score_classB =                         # shape (N_classB,)

scores, labels = process_scores(score_classA, score_classB)

_, _, _, _, auc = calc_ROC_and_AUC(labels, scores, MODE='sklearn')


In [ ]:
bins = np.linspace(scores.min() -1,scores.max()+1, 40)
plt.hist(score_classA, bins=bins, color='#A55194', histtype='step', linewidth=3)
plt.hist(score_classB, bins=bins, color='#8CA252', histtype='step', linewidth=3)

plt.xlabel(r"Score", fontsize=14)
plt.ylabel(r"Counts", fontsize=14)
plt.show()

print("AUC = ", auc)

#### **Question:** What AUC would an AUC < 0.5 mean? What is a score distribution that would lead to this?

In [ ]:
#! YOUR CODE HERE
# score_classA =                         # shape (N_classA,)
# score_classB =                         # shape (N_classB,)

scores, labels = process_scores(score_classA, score_classB)

_, _, _, _, auc = calc_ROC_and_AUC(labels, scores, MODE='sklearn')

In [ ]:
bins = np.linspace(scores.min() -1,scores.max()+1, 40)
plt.hist(score_classA, bins=bins, color='#A55194', histtype='step', linewidth=3)
plt.hist(score_classB, bins=bins, color='#8CA252', histtype='step', linewidth=3)

plt.xlabel(r"Score", fontsize=14)
plt.ylabel(r"Counts", fontsize=14)
plt.show()

print("AUC = ", auc)

Does this mean it's a bad classifier?